### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv
import json
import sqlite3

GROUP = 'gp02'
DATA_URLS = {'observations_dirty.csv': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp02/observations_dirty.csv'}
SCHEMA = {'observations_dirty.csv': {'kind': 'csv', 'fields': ['observation_id', 'category', 'longitude', 'latitude', 'elevation_m', 'status']}}

def valid_cached(path, spec):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if spec["kind"] == "csv":
            with path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.DictReader(handle)
                return set(spec["fields"]).issubset(set(reader.fieldnames or []))
        if spec["kind"] == "geojson":
            document = json.loads(path.read_text(encoding="utf-8"))
            if document.get("type") != "FeatureCollection" or not isinstance(document.get("features"), list):
                return False
            return all(isinstance(feature, dict) and isinstance(feature.get("properties"), dict) and set(spec["fields"]).issubset(feature["properties"]) for feature in document["features"])
        if spec["kind"] == "gpkg":
            if path.read_bytes()[:16] != b"SQLite format 3\x00":
                return False
            with sqlite3.connect(path) as connection:
                if connection.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
                    return False
                layers = {row[0] for row in connection.execute("SELECT table_name FROM gpkg_contents")}
            return set(spec["layers"]).issubset(layers)
    except Exception:
        return False
    return False

def valid_dataset_root(base):
    folder = base / GROUP
    return folder.is_dir() and all(valid_cached(folder / filename, spec) for filename, spec in SCHEMA.items())

candidates = (Path("downloads/python/data"), Path("../../data"), Path("data"))
DATA_ROOT = next((candidate for candidate in candidates if valid_dataset_root(candidate)), None)

if DATA_ROOT is None:
    DATA_ROOT = Path("data")
    target_root = DATA_ROOT / GROUP
    target_root.mkdir(parents=True, exist_ok=True)
    failures = []
    for filename, url in DATA_URLS.items():
        target = target_root / filename
        if valid_cached(target, SCHEMA[filename]):
            print("[OK] Already available:", target)
            continue
        if target.exists():
            target.unlink()
        partial = target.with_name(target.name + ".part")
        if partial.exists():
            partial.unlink()
        try:
            with urlopen(url, timeout=30) as response:
                payload = response.read()
            if not payload:
                raise RuntimeError("empty response")
            partial.write_bytes(payload)
            partial.replace(target)
            if not valid_cached(target, SCHEMA[filename]):
                raise RuntimeError("minimum validation failed")
            print("[OK] Downloaded and validated:", target)
        except Exception as exc:
            if partial.exists():
                partial.unlink()
            if target.exists():
                target.unlink()
            failures.append(filename + ": " + str(exc))
    if failures or not valid_dataset_root(DATA_ROOT):
        raise RuntimeError("Could not prepare dataset " + GROUP + ": " + "; ".join(failures))
    print("[OK] Dataset root:", DATA_ROOT.resolve())
else:
    print("[OK] Using validated dataset root:", (DATA_ROOT / GROUP).resolve())


# GP02 — Clean Geospatial Data

Σε αυτό το notebook θα καθαρίσεις ένα σκόπιμα προβληματικό dataset. Η αρχή είναι: εντοπίζω → κατανοώ → αποφασίζω → τεκμηριώνω → ελέγχω ξανά. Μην διαγράψεις γραμμές πριν μετρήσεις τι χάνεται.

## 1. Load — Φόρτωση και πρώτη επιθεώρηση

Πρώτα μετράμε πριν διορθώσουμε.

In [ ]:
import pandas as pd
import geopandas as gpd
df = pd.read_csv(DATA_ROOT / "gp02/observations_dirty.csv")
print("Rows:", len(df))
print(df.dtypes)
print(df.isna().sum())


## 2. Missing values και data types

Ξεχώρισε essential fields (`observation_id`, longitude, latitude) από optional fields. Το `errors="coerce"` κάνει μη αναγνώσιμες τιμές missing· μετά πρέπει να μετρήσεις τα νέα missing και να αιτιολογήσεις τον κανόνα.

In [ ]:
# Μετέτρεψε τις τρεις στήλες σε αριθμούς.
# Μέτρησε ξανά τα missing values.
for column in ["longitude", "latitude", "elevation_m"]:
    pass


## 3. Duplicates και συντεταγμένες

Αφαίρεσε γραμμές χωρίς observation_id/coordinates, κράτησε μοναδικά IDs και έλεγξε τα όρια EPSG:4326.

In [ ]:
# Συμπλήρωσε τον καθαρισμό.
# clean = ...
# valid_coordinates = ...


## 4. Geometry

Δημιούργησε Points με longitude ως X και latitude ως Y.

In [ ]:
# gdf = gpd.GeoDataFrame(..., crs="EPSG:4326")
# print(gdf.geometry.geom_type.value_counts())
# print(gdf.total_bounds)


## 5. Before / after QA

In [ ]:
# Πρόσθεσε checks για rows, CRS, missing, empty/invalid geometry, μοναδικά IDs και coordinate ranges.
# Κάνε και έναν γρήγορο plot.


## Mini Challenge

Κράτησε ενεργές παρατηρήσεις με elevation_m > 80 και επανέλαβε τους QA checks.

In [ ]:
# selected = ...
# selected.plot(figsize=(7, 6), color="orange")
